# Baseline YOLO model

### Train a baseline YOLO model on the combined mapillary and LISA signs

In [1]:
import os
import csv
import json
import yaml
import shutil
import random
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.notebook import tqdm

## Configuration

In [2]:
# ── Inputs ────────────────────────────────────────────────────────────────────
MAPILLARY_ROOT   = Path("dataset/mtsd_v2_fully_annotated")
LISA_ROOT        = Path("dataset/lisa")
MAPILLARY_CSV    = Path("mapillary_class_review.csv")   # from mapillary review notebook
LISA_CSV         = Path("lisa_review.csv")    # from lisa review notebook
TEST_HOLDOUT = 0.10   # fraction of Mapillary train to hold out as test
USE_TILING        = False

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = Path("dataset/combined_dataset_640")

# ── Split ratios (must sum to 1.0) ────────────────────────────────────────────
# Only applies to Mapillary — LISA keeps its original Roboflow splits
MAPILLARY_SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
MIN_ANNOTATION_PX = 8   # size of smallest annotations to keep

LISA_SPLITS      = ["train", "val", "test"]   # valid -> val in output

SEED             = 42
random.seed(SEED)

## Step 1 — Build unified class list from both CSVs

### Merge similar classes

In [3]:
# Define merges as {canonical_name: [list of names to fold into it]}
# The canonical name is what all variants will become
MAPILLARY_MERGES = {
    "regulatory--no-turn-on-red--g1": [
        "regulatory--no-turn-on-red--g2",
        "regulatory--no-turn-on-red--g3",
    ],
    "warning--divided-highway-ends--g1": [
        "warning--divided-highway-ends--g2",
    ],
    "complementary--one-direction-left--g1": [
        "complementary--one-direction-right--g1",
    ],
    "warning--texts--g1": [
        "warning--texts--g2",
        "warning--texts--g3",
    ]
}

def apply_merges(kept_labels, merges):
    """
    Returns a remapped set of kept labels and a
    lookup {old_label: canonical_label} for the converter to use.
    """
    remap = {}
    for canonical, variants in merges.items():
        for variant in variants:
            if variant in kept_labels:
                remap[variant] = canonical
                kept_labels.discard(variant)
            # Also remap canonical to itself so the converter
            # handles it uniformly
        if canonical in kept_labels:
            remap[canonical] = canonical

    # Pass-through for everything not involved in a merge
    for label in kept_labels:
        if label not in remap:
            remap[label] = label

    return kept_labels, remap

In [4]:
def load_mapillary_kept(csv_path):
    """
    Returns set of mapillary label strings that were marked keep=yes.
    """
    kept = set()
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["keep"] == "yes":
                kept.add(row["label"])
    return kept


def load_lisa_mapping(csv_path):
    """
    Returns {lisa_label: mapillary_label} for kept classes.
    """
    mapping = {}
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["keep"] == "yes":
                mapping[row["lisa_label"]] = row["mapillary_label"]
    return mapping

mapillary_kept  = load_mapillary_kept(MAPILLARY_CSV)
lisa_mapping    = load_lisa_mapping(LISA_CSV)

mapillary_kept, mapillary_remap = apply_merges(mapillary_kept, MAPILLARY_MERGES)

# Union of all target label names (Mapillary labels are the canonical names)
all_labels      = sorted(mapillary_kept | set(lisa_mapping.values()))
label_to_id     = {label: i for i, label in enumerate(all_labels)}

print(f"Mapillary keep classes : {len(mapillary_kept)}")
print(f"LISA keep classes      : {len(lisa_mapping)}")
print(f"Unified class count    : {len(all_labels)}")
print()

# Show any LISA classes that introduce a brand new label not in Mapillary
new_from_lisa = set(lisa_mapping.values()) - mapillary_kept
if new_from_lisa:
    print(f"New classes introduced by LISA ({len(new_from_lisa)}):")
    for n in sorted(new_from_lisa):
        print(f"  {n}")
else:
    print("No new classes introduced by LISA — all map to existing Mapillary labels.")

Mapillary keep classes : 51
LISA keep classes      : 21
Unified class count    : 51

No new classes introduced by LISA — all map to existing Mapillary labels.


## Step 2 — Set up output directory structure

In [5]:
for split in ["train", "val", "test"]:
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_ROOT.resolve()}")

Output directory: C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_640


## Step 3 — Process Mapillary

In [7]:
def load_mapillary_keys(mapillary_root, splits, test_holdout=0.10, seed=42):
    """
    Load and assign each image key to a split.
    Redirects Mapillary's empty test split to val, then carves
    test_holdout fraction of train keys into a proper test set.
    """
    key_to_split = {}
    for split in splits:
        path = mapillary_root / "splits" / f"{split}.txt"
        if not path.exists():
            continue
        with open(path) as f:
            for line in f:
                key = line.strip()
                if key:
                    key_to_split[key] = "val" if split == "test" else split

    train_keys = [k for k, s in key_to_split.items() if s == "train"]
    random.seed(seed)
    random.shuffle(train_keys)
    n_test = int(len(train_keys) * test_holdout)
    for key in train_keys[:n_test]:
        key_to_split[key] = "test"

    from collections import Counter
    counts = Counter(key_to_split.values())
    print(f"  Mapillary split assignment:")
    print(f"    train : {counts['train']:,}")
    print(f"    val   : {counts['val']:,}  (original val + redirected test)")
    print(f"    test  : {counts['test']:,}  ({test_holdout:.0%} held out from train)")

    return key_to_split


def mapillary_to_yolo(bbox, img_w, img_h):
    """Convert Mapillary {xmin,ymin,xmax,ymax} to YOLO cx cy w h (normalised)."""
    xmin, ymin = bbox["xmin"], bbox["ymin"]
    xmax, ymax = bbox["xmax"], bbox["ymax"]
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    w  = (xmax - xmin) / img_w
    h  = (ymax - ymin) / img_h
    return cx, cy, w, h


def is_too_small(bbox, min_px=20):
    """Returns True if the annotation is smaller than min_px in either dimension."""
    w_px = bbox["xmax"] - bbox["xmin"]
    h_px = bbox["ymax"] - bbox["ymin"]
    return w_px < min_px or h_px < min_px

def get_tiles(img_w, img_h, tile_size=640, overlap=0.2):
    """Returns list of (x0, y0, x1, y1) tile coords covering the image."""
    step  = int(tile_size * (1 - overlap))
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1     = min(y0 + tile_size, img_h)
        y0_adj = max(0, y1 - tile_size)
        x0 = 0
        while x0 < img_w:
            x1     = min(x0 + tile_size, img_w)
            x0_adj = max(0, x1 - tile_size)
            tiles.append((x0_adj, y0_adj, x0_adj + tile_size, y0_adj + tile_size))
            if x1 == img_w:
                break
            x0 += step
        if y1 == img_h:
            break
        y0 += step
    return list(dict.fromkeys(tiles))   # deduplicate


def remap_annotations_to_tile(yolo_lines, tile_x0, tile_y0, tile_size,
                                img_w, img_h, min_ann_px=10):
    """
    Takes already-written YOLO lines (strings) in full-image normalised coords
    and remaps them to tile-local coords.
    Only keeps annotations whose centre falls inside the tile.
    """
    tile_x1   = tile_x0 + tile_size
    tile_y1   = tile_y0 + tile_size
    tile_anns = []

    for line in yolo_lines:
        parts          = line.strip().split()
        cls_id         = parts[0]
        cx, cy, bw, bh = [float(x) for x in parts[1:5]]

        # Convert to absolute pixel coords
        pcx = cx * img_w;  pcy = cy * img_h
        pbw = bw * img_w;  pbh = bh * img_h

        # Only keep if centre is inside tile
        if not (tile_x0 <= pcx < tile_x1 and tile_y0 <= pcy < tile_y1):
            continue

        # Clamp box to tile boundary
        pxmin = max(pcx - pbw / 2, tile_x0)
        pymin = max(pcy - pbh / 2, tile_y0)
        pxmax = min(pcx + pbw / 2, tile_x1)
        pymax = min(pcy + pbh / 2, tile_y1)

        if (pxmax - pxmin) < min_ann_px or (pymax - pymin) < min_ann_px:
            continue

        new_cx = max(0.0, min(1.0, (pcx - tile_x0) / tile_size))
        new_cy = max(0.0, min(1.0, (pcy - tile_y0) / tile_size))
        new_bw = max(0.0, min(1.0, (pxmax - pxmin) / tile_size))
        new_bh = max(0.0, min(1.0, (pymax - pymin) / tile_size))

        tile_anns.append(
            f"{cls_id} {new_cx:.6f} {new_cy:.6f} {new_bw:.6f} {new_bh:.6f}"
        )

    return tile_anns

def process_mapillary(mapillary_root, kept_labels, label_to_id,
                      split_ratios, output_root, mapillary_remap=None,
                      MIN_ANNOTATION_PX=32, use_tiling=True):
    ann_dir = mapillary_root / "annotations"
    img_dir = mapillary_root / "images"

    key_to_split = load_mapillary_keys(
        mapillary_root, ["train", "val", "test"],
        test_holdout=TEST_HOLDOUT,
        seed=SEED,
    )
    split_remap = {"train": "train", "val": "val", "test": "test"}

    if mapillary_remap is None:
        mapillary_remap = {label: label for label in kept_labels}

    stats             = defaultdict(lambda: defaultdict(int))
    skipped_no_image  = 0
    skipped_no_labels = 0
    skipped_pano      = 0
    skipped_too_small = 0
    written           = 0

    ann_files = list(ann_dir.glob("*.json"))

    for ann_path in tqdm(ann_files, desc="Mapillary"):
        key = ann_path.stem

        img_path = img_dir / f"{key}.jpg"
        if not img_path.exists():
            skipped_no_image += 1
            continue

        with open(ann_path) as f:
            ann = json.load(f)

        if ann.get("ispano", False):
            skipped_pano += 1
            continue

        img_w = ann["width"]
        img_h = ann["height"]

        yolo_lines = []
        for obj in ann.get("objects", []):
            raw_label = obj.get("label", "")
            if raw_label not in mapillary_remap:
                continue

            props = obj.get("properties", {})
            if props.get("ambiguous") or props.get("out-of-frame") or props.get("dummy"):
                continue

            if is_too_small(obj["bbox"], MIN_ANNOTATION_PX):
                skipped_too_small += 1
                continue

            label    = mapillary_remap[raw_label]
            class_id = label_to_id[label]
            cx, cy, w, h = mapillary_to_yolo(obj["bbox"], img_w, img_h)

            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))

            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            split_out = split_remap.get(key_to_split.get(key, "train"), "train")
            stats[split_out][label] += 1

        if not yolo_lines:
            skipped_no_labels += 1
            continue

        split_out = split_remap.get(key_to_split.get(key, "train"), "train")

        if use_tiling:
            # ── Tile into 640px patches ───────────────────────────────────
            img   = Image.open(img_path).convert("RGB")
            tiles = get_tiles(img_w, img_h, tile_size=640, overlap=0.2)

            wrote_any = False
            for t_idx, (x0, y0, x1, y1) in enumerate(tiles):
                tile_anns = remap_annotations_to_tile(
                    yolo_lines, x0, y0, 640, img_w, img_h, min_ann_px=6
                )
                if not tile_anns:
                    continue

                stem = f"mply_{key}_t{t_idx}"
                img.crop((x0, y0, x1, y1)).save(
                    output_root / split_out / "images" / f"{stem}.jpg", quality=92
                )
                (output_root / split_out / "labels" / f"{stem}.txt").write_text(
                    "\n".join(tile_anns)
                )
                wrote_any = True

            if wrote_any:
                written += 1
        else:
            # ── Copy full image as-is ─────────────────────────────────────
            shutil.copy2(img_path,
                         output_root / split_out / "images" / f"mply_{key}.jpg")
            (output_root / split_out / "labels" / f"mply_{key}.txt").write_text(
                "\n".join(yolo_lines)
            )
            written += 1

    print(f"  Written            : {written:,} images")
    print(f"  Skipped (no image) : {skipped_no_image:,}")
    print(f"  Skipped (pano)     : {skipped_pano:,}")
    print(f"  Skipped (no kept labels) : {skipped_no_labels:,}")
    print(f"  Annotations dropped (too small) : {skipped_too_small:,}")
    return stats

if OUTPUT_ROOT.exists():
    print(f"Clearing existing {OUTPUT_ROOT}...")
    shutil.rmtree(OUTPUT_ROOT)

for split in ["train", "val", "test"]:
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Output directory ready: {OUTPUT_ROOT.resolve()}")

print("Processing Mapillary...")
mapillary_stats = process_mapillary(
    MAPILLARY_ROOT, mapillary_kept, label_to_id,
    MAPILLARY_SPLITS, OUTPUT_ROOT,
    mapillary_remap   = mapillary_remap,
    MIN_ANNOTATION_PX = MIN_ANNOTATION_PX,
    use_tiling        = USE_TILING,
)
print("Done.")

Output directory ready: C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_640
Processing Mapillary...
  Mapillary split assignment:
    train : 32,931
    val   : 15,864  (original val + redirected test)
    test  : 3,658  (10% held out from train)


Mapillary:   0%|          | 0/41909 [00:00<?, ?it/s]

  Written            : 12,138 images
  Skipped (no image) : 0
  Skipped (pano)     : 941
  Skipped (no kept labels) : 28,830
  Annotations dropped (too small) : 78
Done.


## Step 4 — Process LISA

In [8]:
def load_lisa_id_to_name(lisa_root):
    with open(lisa_root / "data.yaml") as f:
        data = yaml.safe_load(f)
    names = data["names"]
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {i: n for i, n in enumerate(names)}


def process_lisa(lisa_root, lisa_mapping, label_to_id,
                 lisa_splits, output_root):
    """
    lisa_mapping: {lisa_label: unified_label}
    """
    lisa_id_to_name = load_lisa_id_to_name(lisa_root)
    split_remap = {"train": "train", "valid": "val", "val": "val", "test": "test"}

    stats = defaultdict(lambda: defaultdict(int))
    skipped_no_image  = 0
    skipped_no_labels = 0
    written = 0

    for split in lisa_splits:
        labels_dir = lisa_root / split / "labels"
        images_dir = lisa_root / split / "images"
        split_out  = split_remap.get(split, split)

        if not labels_dir.exists():
            print(f"  [warn] not found: {labels_dir}")
            continue

        label_files = list(labels_dir.glob("*.txt"))

        for label_file in tqdm(label_files, desc=f"LISA {split}"):
            # Find matching image
            img_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = images_dir / (label_file.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break

            if img_path is None:
                skipped_no_image += 1
                continue

            yolo_lines = []
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue

                    lisa_class_id = int(parts[0])
                    lisa_name     = lisa_id_to_name.get(lisa_class_id)
                    if lisa_name is None or lisa_name not in lisa_mapping:
                        continue  # not a kept class

                    unified_label = lisa_mapping[lisa_name]
                    class_id      = label_to_id[unified_label]

                    cx, cy, w, h  = [float(x) for x in parts[1:5]]
                    cx = max(0.0, min(1.0, cx))
                    cy = max(0.0, min(1.0, cy))
                    w  = max(0.0, min(1.0, w))
                    h  = max(0.0, min(1.0, h))

                    yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                    stats[split_out][unified_label] += 1

            if not yolo_lines:
                skipped_no_labels += 1
                continue

            # Copy image — prefix with lisa_ to avoid key collisions with Mapillary
            out_img = output_root / split_out / "images" / f"lisa_{img_path.name}"
            shutil.copy2(img_path, out_img)

            out_lbl = output_root / split_out / "labels" / f"lisa_{label_file.stem}.txt"
            with open(out_lbl, "w") as f:
                f.write("\n".join(yolo_lines))

            written += 1

    print(f"  Written  : {written:,} images")
    print(f"  Skipped (no image)       : {skipped_no_image:,}")
    print(f"  Skipped (no kept labels) : {skipped_no_labels:,}")
    return stats


print("Processing LISA...")
lisa_stats = process_lisa(
    LISA_ROOT, lisa_mapping, label_to_id,
    LISA_SPLITS, OUTPUT_ROOT
)
print("Done.")

Processing LISA...


LISA train:   0%|          | 0/7956 [00:00<?, ?it/s]

LISA val:   0%|          | 0/1018 [00:00<?, ?it/s]

LISA test:   0%|          | 0/838 [00:00<?, ?it/s]

  Written  : 6,003 images
  Skipped (no image)       : 0
  Skipped (no kept labels) : 3,809
Done.


## Step 5 — Write data.yaml

In [9]:
data_yaml = {
    "path":  str(OUTPUT_ROOT.resolve()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(all_labels),
    "names": all_labels,
}

yaml_out = OUTPUT_ROOT / "data.yaml"
with open(yaml_out, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"data.yaml written to {yaml_out}")

data.yaml written to dataset\combined_dataset_640\data.yaml


## Step 6 — Summary

In [10]:
import numpy as np

def count_files(path):
    return len(list(path.glob("*"))) if path.exists() else 0

print("=" * 70)
print("COMBINED DATASET SUMMARY")
print("=" * 70)
print(f"  Total classes : {len(all_labels)}")
print()

for split in ["train", "val", "test"]:
    n = count_files(OUTPUT_ROOT / split / "images")
    print(f"  {split:<6} images : {n:,}")

print()

# Per-class annotation counts across all splits and both sources
combined_counts = defaultdict(int)
for stats in [mapillary_stats, lisa_stats]:
    for split_counts in stats.values():
        for label, count in split_counts.items():
            combined_counts[label] += count

arr = np.array([combined_counts.get(l, 0) for l in all_labels])

print(f"  Total annotations : {arr.sum():,}")
print(f"  Mean per class    : {arr.mean():,.1f}")
print(f"  Median per class  : {np.median(arr):,.1f}")
print(f"  Classes < 100     : {(arr < 100).sum()}")
print(f"  Classes < 500     : {(arr < 500).sum()}")

print(f"  {'Class':<55} {'Total':>7}  {'Mapillary':>10}  {'LISA':>6}")
print("  " + "─" * 82)

# Flatten per-source totals
mply_totals = defaultdict(int)
for split_counts in mapillary_stats.values():
    for label, count in split_counts.items():
        mply_totals[label] += count

lisa_totals = defaultdict(int)
for split_counts in lisa_stats.values():
    for label, count in split_counts.items():
        lisa_totals[label] += count

for label in all_labels:
    total = combined_counts.get(label, 0)
    mply  = mply_totals.get(label, 0)
    lisa  = lisa_totals.get(label, 0)
    flag  = "  ⚠️ <100" if total < 100 else ""
    print(f"  {label:<55} {total:>7,}  {mply:>10,}  {lisa:>6,}{flag}")

print("=" * 70)

COMBINED DATASET SUMMARY
  Total classes : 51

  train  images : 14,375
  val    images : 2,142
  test   images : 1,624

  Total annotations : 27,142
  Mean per class    : 532.2
  Median per class  : 333.0
  Classes < 100     : 4
  Classes < 500     : 34
  Class                                                     Total   Mapillary    LISA
  ──────────────────────────────────────────────────────────────────────────────────
  complementary--chevron-left--g1                           1,729       1,729       0
  complementary--chevron-right--g1                          1,698       1,698       0
  complementary--keep-left--g1                                333         333       0
  complementary--keep-right--g1                               524          80     444
  complementary--obstacle-delineator--g1                      191         191       0
  complementary--obstacle-delineator--g2                      415         415       0
  complementary--one-direction-left--g1                   

# Training

In [4]:
from ultralytics import YOLO
from pathlib import Path
import os

PROJECT_DIR = os.path.abspath("experiments/")
RUN_NAME    = "baseline_v3_n"

model = YOLO("experiments/baseline_v3_n/weights/last.pt")  # small variant — better than nano for 57 classes

results = model.train(
    # ── Core ──────────────────────────────────────────────────────
    data    = "dataset/combined_dataset_640/data.yaml",
    epochs  = 150,
    patience = 30,           # early stopping — stops if no improvement for 20 epochs
    imgsz   = 640,
    batch   = 16,
    device  = 0,             # GPU; use "cpu" if no GPU available
    resume = True,

    # ── Output ────────────────────────────────────────────────────
    project = PROJECT_DIR,
    name    = RUN_NAME,
    exist_ok = False,        # set True to overwrite a previous run with same name

    # ── Optimizer ─────────────────────────────────────────────────
    optimizer = "SGD",       # SGD generalizes better than Adam for detection
    lr0       = 0.01,        # initial learning rate
    lrf       = 0.01,        # final lr = lr0 * lrf (cosine decay target)
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    cos_lr    = True,        # cosine LR schedule — smoother decay than step
    cls = 0.5,
    box = 7.5,

    # ── Augmentation ──────────────────────────────────────────────
    hsv_h     = 0.015,       # hue jitter — helps with varied lighting
    hsv_s     = 0.7,         # saturation jitter
    hsv_v     = 0.4,         # brightness jitter — important for day/night variation
    degrees   = 5.0,         # small rotation — signs are mostly upright
    translate = 0.1,
    scale     = 0.7,         # scale jitter — critical for far/close sign detection
    shear     = 2.0,
    perspective = 0.0003,    # slight perspective for dashcam angle variation
    flipud    = 0.0,         # don't flip vertically — signs aren't upside down
    fliplr    = 0.0,         # don't flip horizontally — no/left/right turns would become wrong
    mosaic    = 1.0,         # combine 4 images — helps with small/far signs
    mixup     = 0.10,         # blend two images — improves generalization
    copy_paste = 0.3,        # paste objects into scenes — helps rare classes
    copy_paste_mode = 'mixup', # copy paste from other classes instead of the default
    close_mosaic = 10,

    # ── Evaluation ────────────────────────────────────────────────
    val       = True,
    save      = True,
    save_period = 10,        # save checkpoint every 10 epochs
    plots     = True,        # generate confusion matrix and metric plots
)

New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=mixup, cos_lr=True, cutmix=0.0, data=dataset/combined_dataset_640/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=experiments\baseline_v3_n\weights\last.pt, mome

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO

model = YOLO("experiments/baseline_v3_n/weights/best.pt")

metrics = model.val(
    data="dataset/combined_dataset_640/data.yaml",
    split="test",
    imgsz=640,
    device=0,
    plots=True,
    save_json=False,
)

print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
Model summary (fused): 73 layers, 3,015,593 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 718.6572.0 MB/s, size: 749.3 KB)
val: Scanning C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_640\test\labels.cache... 1624 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1624/1624  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 12.4it/s 8.3s0.1s
                   all       1624       2400       0.59      0.447      0.471      0.338
complementary--chevron-left--g1         65        158      0.678      0.589       0.63      0.386
complementary--chevron-right--g1         61        163      0.691      0.536      0.585      0.343
complementary--keep-left--g1         28         31      0.589      0.581      0.574      0.313
complementary--keep-right--g1   

### Evaluate the model and find confused classes

In [1]:
import numpy as np
import json
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_PATH   = "experiments/baseline_v3/weights/best.pt"
DATA_YAML    = "dataset/combined_dataset_640/data.yaml"
SPLIT        = "val"
CONF         = 0.15
IOU          = 0.45
DEVICE       = 0
TOP_N        = 30      # show top N most confused pairs
MIN_COUNT    = 5       # ignore confusion pairs with fewer than this many instances

# ── Run validation to get confusion matrix ────────────────────────────────────
model   = YOLO(MODEL_PATH)
metrics = model.val(
    data    = DATA_YAML,
    split   = SPLIT,
    conf    = CONF,
    iou     = IOU,
    device  = DEVICE,
    plots   = True,
    verbose = False,
)

# ── Extract confusion matrix ──────────────────────────────────────────────────
# Ultralytics stores the raw confusion matrix on the validator object
cm         = metrics.confusion_matrix.matrix   # shape: (nc+1, nc+1)
class_names = list(model.names.values())       # ordered list of class names
nc         = len(class_names)

# Last row/col is background (FP/FN)
# cm[i][j] = number of times class j was predicted when true class was i

# ── Build confusion pairs ─────────────────────────────────────────────────────
pairs = []

for true_idx in range(nc):
    true_name  = class_names[true_idx]
    row_total  = cm[true_idx, :nc].sum()   # total true instances (excl background)
    if row_total == 0:
        continue

    for pred_idx in range(nc):
        if pred_idx == true_idx:
            continue   # skip correct predictions
        count = int(cm[true_idx, pred_idx])
        if count < MIN_COUNT:
            continue

        pred_name  = class_names[pred_idx]
        confusion_rate = count / row_total if row_total > 0 else 0

        pairs.append({
            "true"          : true_name,
            "predicted"     : pred_name,
            "count"         : count,
            "true_total"    : int(row_total),
            "confusion_rate": confusion_rate,
        })

# Sort by count descending
pairs.sort(key=lambda x: x["count"], reverse=True)

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  TOP {TOP_N} CONFUSED CLASS PAIRS  (min {MIN_COUNT} instances)")
print(f"{'='*80}")
print(f"  {'True Class':<45} {'Predicted As':<45} {'Count':>6}  {'Rate':>6}")
print(f"  {'─'*45} {'─'*45} {'─'*6}  {'─'*6}")

for p in pairs[:TOP_N]:
    print(f"  {p['true']:<45} {p['predicted']:<45} "
          f"{p['count']:>6}  {p['confusion_rate']:>5.1%}")

# ── Background confusion (missed detections) ──────────────────────────────────
print(f"\n{'='*80}")
print(f"  CLASSES MOST OFTEN MISSED (predicted as background)")
print(f"{'='*80}")
print(f"  {'Class':<50} {'Missed':>7}  {'Total':>7}  {'Miss Rate':>9}")
print(f"  {'─'*50} {'─'*7}  {'─'*7}  {'─'*9}")

missed = []
for true_idx in range(nc):
    missed_count = int(cm[true_idx, nc])   # last col = predicted background
    row_total    = int(cm[true_idx, :].sum())
    if row_total == 0 or missed_count < MIN_COUNT:
        continue
    missed.append({
        "class"      : class_names[true_idx],
        "missed"     : missed_count,
        "total"      : row_total,
        "miss_rate"  : missed_count / row_total,
    })

missed.sort(key=lambda x: x["missed"], reverse=True)
for m in missed[:TOP_N]:
    print(f"  {m['class']:<50} {m['missed']:>7}  {m['total']:>7}  "
          f"{m['miss_rate']:>8.1%}")

# ── Per-superclass confusion summary ─────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  CONFUSION BY SUPERCLASS")
print(f"{'='*80}")

def get_superclass(label):
    if label.startswith("regulatory--maximum-speed-limit"):
        return "speed_limit"
    prefix = label.split("--")[0]
    return prefix

superclass_confusion = defaultdict(lambda: defaultdict(int))

for p in pairs:
    true_sc = get_superclass(p["true"])
    pred_sc = get_superclass(p["predicted"])
    if true_sc == pred_sc:
        superclass_confusion[true_sc]["within"] += p["count"]
    else:
        superclass_confusion[true_sc]["cross"]  += p["count"]

print(f"  {'Superclass':<25} {'Within-class':>13}  {'Cross-class':>12}  {'Within %':>9}")
print(f"  {'─'*25} {'─'*13}  {'─'*12}  {'─'*9}")
for sc, counts in sorted(superclass_confusion.items()):
    within  = counts["within"]
    cross   = counts["cross"]
    total   = within + cross
    pct     = within / total if total > 0 else 0
    print(f"  {sc:<25} {within:>13,}  {cross:>12,}  {pct:>8.1%}")

# ── Plot: top confused pairs bar chart ────────────────────────────────────────
top_pairs = pairs[:15]
labels    = [f"{p['true'].split('--')[1][:20]}\n→{p['predicted'].split('--')[1][:20]}"
             for p in top_pairs]
counts    = [p["count"] for p in top_pairs]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(labels[::-1], counts[::-1], color="#e07b54", edgecolor="white")
ax.set_xlabel("Confusion count")
ax.set_title("Top Confused Class Pairs")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=8)
plt.tight_layout()
plt.savefig("confusion_pairs.png", dpi=100, bbox_inches="tight")
plt.show()
print("\nChart saved to confusion_pairs.png")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
Model summary (fused): 73 layers, 11,145,321 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 432.6333.7 MB/s, size: 321.9 KB)
val: Scanning C:\Projects\TrafficSignRecognition\model_training\dataset\combined_dataset_640\val\labels.cache... 2142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2142/2142  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 11.3it/s 11.9s0.2s
                   all       2142       3287      0.735      0.473      0.612      0.469
Speed: 0.9ms preprocess, 1.9ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to C:\Projects\TrafficSignRecognition\model_training\experiments\detect\val8

  TOP 30 CONFUSED CLASS PAIRS  (min 5 instances)
  True Class                                    Predicted As                                   Coun

<Figure size 1400x600 with 1 Axes>


Chart saved to confusion_pairs.png


### Export to ONNX

In [1]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH  = "experiments/baseline_v3/weights/best.pt"
EXPORT_IMGSZ = 640

model = YOLO(MODEL_PATH)

export_path = model.export(
    format  = "onnx",
    imgsz   = EXPORT_IMGSZ,
    half    = False,    # ONNX doesn't support FP16 on all runtimes — keep FP32
    dynamic = False,    # static input shape — required for most Android ONNX runtimes
    simplify = True,    # simplify the ONNX graph — improves compatibility and speed
)

print(f"Exported to: {export_path}")
print(f"File size  : {Path(export_path).stat().st_size / 1e6:.1f} MB")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
Model summary (fused): 73 layers, 11,145,321 parameters, 0 gradients, 28.5 GFLOPs

PyTorch: starting from 'experiments\baseline_v3\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 55, 8400) (21.5 MB)

ONNX: starting export with onnx 1.20.1 opset 22...


c:\Projects\TrafficSignRecognition\model_training\.venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.89...
ONNX: export success  1.4s, saved as 'experiments\baseline_v3\weights\best.onnx' (42.7 MB)

Export complete (1.7s)
Results saved to C:\Projects\TrafficSignRecognition\model_training\experiments\baseline_v3\weights
Predict:         yolo predict task=detect model=experiments\baseline_v3\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=experiments\baseline_v3\weights\best.onnx imgsz=640 data=dataset/combined_dataset_640/data.yaml  
Visualize:       https://netron.app
Exported to: experiments\baseline_v3\weights\best.onnx
File size  : 44.8 MB


### Generate labels file

In [2]:
import yaml
import json
from pathlib import Path

with open("dataset/combined_dataset_640/data.yaml") as f:
    data = yaml.safe_load(f)

# Save as plain text — one class per line, index = line number
with open("classes.txt", "w") as f:
    for name in data["names"]:
        f.write(name + "\n")

# Also save as JSON in case they prefer it
with open("classes.json", "w") as f:
    json.dump({i: name for i, name in enumerate(data["names"])}, f, indent=2)

print(f"{len(data['names'])} classes exported")
print("Files: classes.txt, classes.json")

51 classes exported
Files: classes.txt, classes.json
